In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType, TimestampType, IntegerType
import pyspark.sql.functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_procedure")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Procedure/"
silver_procedure_bundle_path = "../../data_lake/silver/silver_procedure_bundle/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])

time_schema = StructType([
        StructField("start", TimestampType(), True),
        StructField("end", TimestampType(), True)
    ])

reference_schema = StructType([
            StructField("reference", StringType(), True),
            StructField("display", StringType(), True)
        ])

In [ ]:
df_bronze = spark.read.format("parquet").load(bronze_path)

In [ ]:
df_inter = df_bronze.select(
    col("resource.id").alias("procedure_id"),
    col("resource.status").alias("status"),
    from_json(col("resource.code"), type_schema).alias("code"),
    get_json_object(col("resource.subject"), "$.reference").alias("patient_id"),
    get_json_object(col("resource.encounter"), "$.reference").alias("encounter_id"),
    from_json(col("resource.performedPeriod"), time_schema).alias("performed_time"),
    from_json(col("resource.location"), reference_schema).alias("location"),
    from_json(col("resource.reasonreference"), ArrayType(reference_schema)).alias("reasonreference"),
    from_json(col("resource.reasonCode"), ArrayType(type_schema)).alias("reasoncode"),
    col("input_file_name")
)

In [ ]:
df_inter2 = df_inter.select(
    col("procedure_id"),
    col("status"),
    col("code")["coding"][0]["display"].alias("code_display"),
    col("code")["coding"][0]["code"].alias("code"),
    F.split(col("patient_id"), ":").getItem(2).alias("patient_id"),
    F.split(col("encounter_id"), ":").getItem(2).alias("encounter_id"),
    col("performed_time")["start"].alias("performed_start_time"),
    col("performed_time")["end"].alias("performed_end_time"),
    F.split(col("location")["reference"], "\|").getItem(1).alias("location_id"),
    F.split(col("reasonreference")[0]["reference"], ":").getItem(2).alias("reason_reference"),
    col("reasonreference")[0]["display"].alias("reason_reference_display"),
    col("reasoncode")[0]["coding"][0]["display"].alias("reason_code_display"),
    col("reasoncode")[0]["coding"][0]["code"].alias("reason_code"),
    col("input_file_name")
)

In [ ]:
df_inter2.write.mode("overwrite").format("parquet").save(silver_procedure_bundle_path)

In [ ]:
spark.stop()